# 08 — Model Comparison

Load all four trained models, run evaluation on the same test set, and produce a unified comparison of ROC-AUC, AUPRC, confusion matrices, and PR curves.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

from src.data_preprocessing import load_processed
from src.utils import load_model
from src.evaluation import evaluate_model, save_metrics_comparison
from src.visualization import (
    plot_confusion_matrix, plot_roc_curves, plot_pr_curves, plot_metrics_bar
)

In [ ]:
X_train, X_test, y_train, y_test = load_processed()

## Evaluate All Models

In [ ]:
model_names = ['logistic_regression', 'decision_tree', 'random_forest', 'xgboost']
results = []

for name in model_names:
    model = load_model(name)
    result = evaluate_model(model, X_test, y_test, name)
    results.append(result)

## Confusion Matrices

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, r in zip(axes, results):
    disp = ConfusionMatrixDisplay(confusion_matrix=r['confusion_matrix'],
                                   display_labels=['Legit', 'Fraud'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(r['model_name'].replace('_', ' ').title())
plt.suptitle('Confusion Matrices — All Models', y=1.02)
plt.tight_layout()
plt.show()

## ROC Curves

In [ ]:
plot_roc_curves(results)

## Precision-Recall Curves

In [ ]:
plot_pr_curves(results)

## Metrics Summary

In [ ]:
metrics_df = save_metrics_comparison(results)
plot_metrics_bar(metrics_df)

## Conclusion

| Model | Strength | Weakness |
|---|---|---|
| Logistic Regression | Fast, interpretable, good baseline | May under-fit complex patterns |
| Decision Tree | Interpretable splits | Prone to overfitting, lower AUPRC |
| Random Forest | High recall, robust | Slower inference, less interpretable |
| XGBoost | Best AUPRC, handles imbalance natively | Requires tuning, black-box |

**Recommended model: XGBoost** — highest AUPRC and F1 score on fraud class, with native imbalance handling via `scale_pos_weight`.